# Bài học 17: Team và xử lý hàng loạt

Trong 2 bài trước, chúng ta đã xây dựng từng agent riêng lẻ: Content Writer (Bài 15), Image Finder và AIO Analyzer (Bài 16). Giờ là lúc **nối tất cả lại** thành một team phối hợp:

```
Yêu cầu của bạn --> [Team Leader] --> phân công cho --> [Content Writer / Image Finder / AIO Analyzer]
```

Đây chính là **kiến trúc sản phẩm thật**. Chúng ta import trực tiếp từ source code.

Điểm khác biệt so với việc chạy từng agent riêng lẻ:
- **Team Leader** — đọc yêu cầu và phân công cho đúng thành viên
- **Xử lý song song** (parallel processing) — tạo nhiều bài viết cùng lúc
- **Task mode** — team có thể chạy nhiều nhiệm vụ đồng thời

## Agno Team

Agno **Team** là một nhóm các agent được quản lý bởi một leader:

```
Team Leader (Claude Sonnet) -- đọc yêu cầu, quyết định ai xử lý
  |
  +-- Content Writer -- nghiên cứu và viết bài
  +-- Image Finder -- tìm và chèn hình ảnh (tùy chọn)
  +-- AIO Analyzer -- phân tích Google AI Overviews
```

Khái niệm quan trọng: **TeamMode.tasks** cho phép leader chạy nhiều nhiệm vụ song song.

Khi bạn nói “Tạo bài viết về Chủ đề A, Chủ đề B, Chủ đề C”, leader tạo 3 task song song thay vì xử lý từng cái một.

## Xử lý lỗi: try / except

Điều gì xảy ra khi có sự cố? Lời gọi API thất bại, tìm kiếm không trả về kết quả, hoặc mạng bị mất.

Python dùng **try/except** để xử lý lỗi một cách an toàn:

```python
try:
    # Thử chạy đoạn code này
    result = agent.run("...")
except Exception as e:
    # Nếu thất bại, làm cái này thay thế
    print(f"Đã xảy ra lỗi: {e}")
```

- Code bên trong `try` chạy bình thường
- Nếu bất kỳ dòng nào trong `try` thất bại, Python **nhảy** sang `except` ngay lập tức
- Biến `e` chứa thông báo lỗi
- Nếu không có try/except, toàn bộ chương trình sẽ bị crash

**Một bài viết lỗi không làm crash cả batch** — các bài viết khác vẫn tiếp tục xử lý bình thường.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## Định nghĩa Team

Hãy xem code team thật từ `output/backend/agents/team.py`:

In [ ]:
# Show the actual team definition
team_path = os.path.abspath("../../output/backend/agents/team.py")
with open(team_path, "r", encoding="utf-8") as f:
    print(f.read())

Những điểm chính từ định nghĩa team:

- `id="seo-workspace"` — định danh team trong API
- `mode=TeamMode.tasks` — bật chế độ chạy task song song
- `members=[content_writer, aio_analyzer]` + tùy chọn `image_finder`
- **Image Finder là tùy chọn** — chỉ được thêm vào nếu DataForSEO đã được cài đặt
- `db=SqliteDb(...)` — lịch sử chat được lưu trong SQLite
- `store_member_responses=True` — giữ lại câu trả lời của thành viên để làm ngữ cảnh

## Tạo một bài viết

Trước tiên, tạo một bài viết thông qua team:

> **Chi phí:** ~$0.20-0.50 mỗi bài viết (gọi Sonnet cho team leader + content writer). Mất khoảng 1-2 phút.

In [ ]:
from agents.team import team

topic = "10 Simple SEO Tips for Beginners"
print(f"Creating article: {topic}\n")

response = team.run(f"Write an SEO article about: {topic}")
print(response.content[:2000])
if len(response.content) > 2000:
    print("\n... (truncated)")

In [ ]:
# Check what was saved
import json
from tools.storage import list_all_articles, get_article_content

result = list_all_articles()
articles = json.loads(result)
print(f"Articles in storage: {len(articles)}")
for a in articles:
    print(f"  {a['id']}: {a['topic']} ({a['word_count']} words, {a['status']})")

## Đọc file trong Python

Bài viết được lưu dưới dạng file `.md` trong thư mục `content/`:

```python
with open("filename.md", "r", encoding="utf-8") as f:
    content = f.read()
```

Giải thích từng phần:
- `open(path, "r")` — mở file để **đọc** (`"r"` = read, `"w"` sẽ là để **ghi**)
- `encoding="utf-8"` — xử lý đúng các ký tự đặc biệt (dấu tiếng Việt, emoji, v.v.)
- `with ... as f:` — khối `with` tự động đóng file khi xong
- `f.read()` — đọc toàn bộ nội dung file vào một chuỗi

In [ ]:
# Read the saved article file
if articles:
    last_article = articles[-1]
    content_json = get_article_content(last_article['id'])
    content = json.loads(content_json)
    print(f"Article: {content['topic']}\n")
    print(content['article_markdown'][:2000])
    if len(content.get('article_markdown', '')) > 2000:
        print("\n... (truncated for display)")

## Xử lý hàng loạt — nhiều bài viết song song

Sức mạnh thật sự của `TeamMode.tasks`: tạo **nhiều bài viết cùng lúc**.

Thay vì tạo bài viết từng cái một (tuần tự), team chạy chúng song song. Với 3 bài viết, tốc độ nhanh gấp ~3 lần.

> **Chi phí:** ~$0.60-1.50 cho 3 bài viết. Mất khoảng 2-4 phút tổng cộng (so với 6-12 phút nếu chạy tuần tự).

In [ ]:
import time

topics = [
    "What is Technical SEO and Why It Matters",
    "How to Write Meta Descriptions That Get Clicks",
    "Internal Linking Strategy for Better Rankings",
]

print(f"Creating {len(topics)} articles in batch...\n")
start = time.time()

# Send all topics in one prompt — the team leader creates parallel tasks
prompt = "Write SEO articles about each of these topics:\n"
for i, t in enumerate(topics, 1):
    prompt += f"{i}. {t}\n"

response = team.run(prompt)

elapsed = time.time() - start
print(f"\nDone in {elapsed:.0f} seconds")
print(response.content[:3000])

In [ ]:
# See the results
result = list_all_articles()
articles = json.loads(result)
print(f"Total articles in storage: {len(articles)}\n")

total_words = 0
for a in articles:
    wc = a.get('word_count', 0) or 0
    total_words += wc
    print(f"  {a['id']}: {a['topic']}")
    print(f"    {wc} words, status: {a['status']}")

print(f"\nTotal words across all articles: {total_words:,}")

## Bài tập

Sử dụng danh sách `articles` ở trên, kiểm tra chất lượng bài viết:

1. Số từ có trên 1000 cho mỗi bài không? In ra đạt/không đạt cho từng bài.
2. Các bài viết đang ở trạng thái gì? (nên là `"review"` nếu thành công)
3. Đếm số bài viết ở mỗi trạng thái.

In [ ]:
# Bài tập: Điền vào chỗ trống để kiểm tra chất lượng bài viết

# 1. Kiểm tra số từ cho mỗi bài viết
for a in articles:
    wc = a.get(___, 0) or 0
    status = "ĐẠT" if wc > ___ else "KHÔNG ĐẠT"
    print(f"{status}: {a['topic']} ({wc} từ)")

# 2. In trạng thái của mỗi bài viết
print("\nTrạng thái bài viết:")
for a in articles:
    print(f"  {a['id']}: {a[___]}")

# 3. Đếm số bài viết theo trạng thái
status_counts = {}
for a in articles:
    s = a["status"]
    status_counts[s] = status_counts.get(s, 0) + ___
print(f"\nSố lượng theo trạng thái: {status_counts}")

<details>
<summary>Bấm để xem đáp án</summary>

```python
# 1. Kiểm tra số từ cho mỗi bài viết
for a in articles:
    wc = a.get("word_count", 0) or 0
    status = "ĐẠT" if wc > 1000 else "KHÔNG ĐẠT"
    print(f"{status}: {a['topic']} ({wc} từ)")

# 2. In trạng thái của mỗi bài viết
print("\nTrạng thái bài viết:")
for a in articles:
    print(f"  {a['id']}: {a['status']}")

# 3. Đếm số bài viết theo trạng thái
status_counts = {}
for a in articles:
    s = a["status"]
    status_counts[s] = status_counts.get(s, 0) + 1
print(f"\nSố lượng theo trạng thái: {status_counts}")
```
</details>

## Tổng kết

Một **team SEO bằng AI** hoàn chỉnh với xử lý hàng loạt:

- **3 agent chuyên biệt**, mỗi agent dùng Claude Sonnet với các tools riêng
- **Team Leader** phân công yêu cầu cho đúng thành viên
- **Xử lý hàng loạt** — nhiều bài viết được tạo song song
- **Lưu trữ cục bộ** — bài viết lưu dưới dạng file `.md` với metadata trong JSON

Code này chạy **y hệt** trong sản phẩm thật. Khi bạn chạy:
```bash
python output/backend/serve.py
```
Và gửi prompt qua giao diện web, nó dùng đúng team mà chúng ta vừa kiểm thử ở đây.

### Tiếp theo: Mô-đun 5

Mô-đun tiếp theo hoàn thiện sản phẩm với:
- Tầng lưu trữ cục bộ — cách bài viết được lưu trữ
- Cách mọi thứ kết nối — toàn bộ kiến trúc
- Giao diện web — chat qua trình duyệt
- Mở rộng sản phẩm — thêm khả năng mới bằng vibecoding